# End-to-End EDA + Visualization + Modeling Notebook

This notebook connects to the Postgres warehouse, runs exploratory data analysis, creates features, trains a model, and evaluates performance.

In [1]:
# 1. Setup and imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

sns.set(style="whitegrid")

# PostgreSQL connection string - adjust user/password as needed
engine = create_engine("postgresql://huseyn@localhost:5432/b2b_credit_risk")

ModuleNotFoundError: No module named 'sqlalchemy'

In [ ]:
# 2. Load data from Postgres (via DBeaver connection credentials)
query= """
SELECT * FROM credit_risk_dw.fact_exposure_snapshot LIMIT 10;
"""
print(pd.read_sql(query, engine).head())

# Pivots of key tables used in modeling
fact_exposure = pd.read_sql("SELECT * FROM credit_risk_dw.fact_exposure_snapshot", engine)
fact_default = pd.read_sql("SELECT * FROM credit_risk_dw.fact_default_event", engine)
dim_customer = pd.read_sql("SELECT * FROM credit_risk_dw.dim_customer", engine)

In [ ]:
# 3. EDA summaries
print("fact_exposure", fact_exposure.shape)
print("fact_default", fact_default.shape)
print("dim_customer", dim_customer.shape)

print(fact_exposure.describe())
print(fact_default['default_amount'].describe())

# distribution of target: defaults by customer
cust_defaults = fact_default.groupby('customer_key').size().reset_index(name='n_defaults')
print(cust_defaults.n_defaults.value_counts())

In [ ]:
# 4. Visualizations
plt.figure(figsize=(10,5))
sns.histplot(fact_exposure['utilization_ratio'], bins=50, kde=True)
plt.title('Utilization ratio distribution')
plt.show()

plt.figure(figsize=(10,5))
sns.boxplot(data=fact_exposure, x='rating_key', y='overdue_exposure')
plt.title('Overdue exposure by rating')
plt.show()

In [ ]:
# 5. Feature engineering
# Use phase 2 model features and match to default label (simplified)
features = fact_exposure.groupby('customer_key').agg(
    current_exposure=('current_exposure','mean'),
    overdue_exposure=('overdue_exposure','mean'),
    utilization_ratio=('utilization_ratio','mean'),
    avg_sales=('monthly_sales_estimate','mean'),
    invoice_count=('invoice_count_month','mean'),
    rating_key=('rating_key','max')
).reset_index()

labels = (fact_default.groupby('customer_key')
          .size().reset_index(name='has_default'))
labels['has_default'] = (labels['has_default'] > 0).astype(int)

data = features.merge(labels, on='customer_key', how='left').fillna(0)
print(data.head())

In [ ]:
# 6. Training and evaluation
X = data[['current_exposure','overdue_exposure','utilization_ratio','avg_sales','invoice_count','rating_key']]
y = data['has_default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_proba))

In [ ]:
# 7. Save model and results
import joblib
joblib.dump(model, 'models/eda_rf_default_model.pkl')

print('Saved model to models/eda_rf_default_model.pkl')